# Fine-tuning — Joint Aspect-Sentiment DeBERTa

In [1]:
# # ============================================================
# # 0. Install packages for Google Colab
# # ============================================================
# # Runtime recommendation:
# # Runtime → Change runtime type → GPU → T4

# INSTALL_PACKAGES = True

# if INSTALL_PACKAGES:
#     %pip install -q -U transformers accelerate tokenizers safetensors sentencepiece protobuf scikit-learn pandas numpy matplotlib tqdm ipywidgets

In [2]:
# # ============================================================
# # 1. Mount Google Drive
# # ============================================================

# from google.colab import drive
# drive.mount("/content/drive")

In [2]:
# ============================================================
# 2. Imports and global settings
# ============================================================

import os
import gc
import json
import random
import shutil
import inspect
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    hamming_loss,
    classification_report,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    try:
        print(subprocess.check_output(["nvidia-smi"], text=True))
    except Exception as e:
        print("nvidia-smi check skipped:", e)
else:
    print("WARNING: GPU is not available. Change Colab runtime to GPU/T4 before training.")

Device: cuda
Torch: 2.12.1+cu126
Torch CUDA build: 12.6
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Wed Jun 24 14:14:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.62                 KMD Version: 610.62        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   44C    P0             13W /   60W |       0MiB /   4096MiB |      0%      Default |
|                                         | 

In [12]:
# ============================================================
# 3. Path configuration for Google Colab
# ============================================================
# Local equivalent:
# C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\
#
# Colab equivalent used here:
# /content/drive/MyDrive/Skripsi/dataset/phase1/trustpilot_reformatted_output/

DRIVE_ROOT = Path("C:\\Users\\Lenovo\\Documents")

PROJECT_ROOT = DRIVE_ROOT / "Skripsi"

DATA_DIR = PROJECT_ROOT / "dataset" / "phase1" / "trustpilot_reformatted_output"
DATA_PATH = DATA_DIR / "01_trustpilot_reformatted_wide_light_preprocessed.csv"

CANDIDATE_DATA_PATHS = [
    DATA_PATH,
    DRIVE_ROOT / "SkripsiColabs" / "Phase1" / "trustpilot_reformatted_output" / "01_trustpilot_reformatted_wide_light_preprocessed.csv",
    DRIVE_ROOT / "SkripsiColabs" / "Phase1" / "Output" / "01_trustpilot_reformatted_wide_light_preprocessed.csv",
    Path("/content/01_trustpilot_reformatted_wide_light_preprocessed.csv"),
]

for candidate in CANDIDATE_DATA_PATHS:
    if candidate.exists():
        DATA_PATH = candidate
        DATA_DIR = DATA_PATH.parent
        break

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find 01_trustpilot_reformatted_wide_light_preprocessed.csv. "
        "Upload it to Drive or edit DATA_PATH manually. Current DATA_PATH: "
        f"{DATA_PATH}"
    )

OUTPUT_DIR = DATA_DIR / "deberta_joint_aspect_sentiment_output_colab_base2"
RUNS_DIR = OUTPUT_DIR / "tuning_runs2"
BEST_MODEL_DIR = OUTPUT_DIR / "colab_best_joint_deberta_model2"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("RUNS_DIR:", RUNS_DIR)

DATA_PATH: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\01_trustpilot_reformatted_wide_light_preprocessed.csv
OUTPUT_DIR: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base2
RUNS_DIR: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base2\tuning_runs2


In [13]:
# ============================================================
# 4. Load dataset
# ============================================================

df = pd.read_csv(DATA_PATH)

print("Dataset path:", DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head(3))
print(df.columns.tolist())

Dataset path: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\01_trustpilot_reformatted_wide_light_preprocessed.csv
Dataset shape: (2782, 28)


,Card_id,Rating,Location,Date,Total Review,shop,Year,General,General_text,text_raw,...,Pricing,Product_availability,Payment,Packaging,Delivered_product_status,Shipping,Return_process,Refund_process,Customer_service,Trust
0,591ed0daa02b420900675219,5,Britain (UK),2017-05-19,1 review,Zalando,2017,1.0,Positive,Excellent and well organised. Excellent and we...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,5b7af35f14b7d805ec7389c4,1,Britain (UK),2018-08-20,3 reviews,Zalando,2018,0.0,Negative,Bizarre returns service.,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,5fe77011755dc107e0cf497f,1,Finland,2020-12-26,1 review,Zalando,2020,0.0,Negative,I have ordered jacket and then returned…. I ha...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0


['Card_id', 'Rating', 'Location', 'Date', 'Total Review', 'shop', 'Year', 'General', 'General_text', 'text_raw', 'text_clean', 'aspect_count', 'positive_aspect_count', 'negative_aspect_count', 'Information', 'Product_features', 'Item_quality', 'App_experience', 'Pricing', 'Product_availability', 'Payment', 'Packaging', 'Delivered_product_status', 'Shipping', 'Return_process', 'Refund_process', 'Customer_service', 'Trust']


In [14]:
# ============================================================
# 5. Build joint aspect-sentiment multilabel dataset
# ============================================================

ASPECT_COLS = [
    "Information",
    "Product_features",
    "Item_quality",
    "App_experience",
    "Pricing",
    "Product_availability",
    "Payment",
    "Delivered_product_status",
    "Shipping",
    "Return_process",
    "Refund_process",
    "Customer_service",
    "Trust",
]

TEXT_COL = "text_clean"

missing_cols = [c for c in ASPECT_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing aspect columns: {missing_cols}")

if TEXT_COL not in df.columns:
    raise ValueError(f"Text column '{TEXT_COL}' was not found.")

work_df = df.copy()

JOINT_LABEL_COLS = []

for aspect in ASPECT_COLS:
    pos_col = f"{aspect}_Positive"
    neg_col = f"{aspect}_Negative"

    # Original Trustpilot label meaning:
    # NaN = aspect absent
    # 1   = positive aspect sentiment
    # 0   = negative aspect sentiment
    work_df[pos_col] = (work_df[aspect] == 1).astype(int)
    work_df[neg_col] = (work_df[aspect] == 0).astype(int)

    JOINT_LABEL_COLS.extend([pos_col, neg_col])

work_df[TEXT_COL] = work_df[TEXT_COL].fillna("").astype(str).str.strip()
work_df["joint_label_count"] = work_df[JOINT_LABEL_COLS].sum(axis=1)

model_df = work_df[
    (work_df[TEXT_COL] != "") &
    (work_df["joint_label_count"] > 0)
].copy().reset_index(drop=True)

print("Original rows:", len(df))
print("Rows usable for joint model:", len(model_df))
print("Number of aspects:", len(ASPECT_COLS))
print("Number of joint labels:", len(JOINT_LABEL_COLS))

label_counts = model_df[JOINT_LABEL_COLS].sum().sort_values(ascending=False)

display(label_counts.to_frame("positive_count"))

JOINT_DATASET_PATH = OUTPUT_DIR / "11_trustpilot_joint_aspect_sentiment_multilabel_colab.csv"

columns_to_save = [
    "Card_id", "Rating", "Location", "Date", "Total Review", "shop", "Year",
    "General", "General_text", "text_raw", "text_clean", "aspect_count",
    "positive_aspect_count", "negative_aspect_count", "joint_label_count",
] + ASPECT_COLS + JOINT_LABEL_COLS

columns_to_save = [c for c in columns_to_save if c in model_df.columns]

model_df[columns_to_save].to_csv(
    JOINT_DATASET_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved joint dataset:", JOINT_DATASET_PATH)

Original rows: 2782
Rows usable for joint model: 2782
Number of aspects: 13
Number of joint labels: 26


,positive_count
Shipping_Positive,873
Trust_Positive,744
Shipping_Negative,642
Item_quality_Positive,628
Trust_Negative,437
Pricing_Positive,416
Customer_service_Negative,341
Customer_service_Positive,308
Product_availability_Positive,218
Refund_process_Negative,188


Saved joint dataset: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base2\11_trustpilot_joint_aspect_sentiment_multilabel_colab.csv


In [15]:
# ============================================================
# 6. Train / validation / test split
# ============================================================

train_df, temp_df = train_test_split(
    model_df,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

split_summary = pd.DataFrame({
    "label": JOINT_LABEL_COLS,
    "train_count": train_df[JOINT_LABEL_COLS].sum().astype(int).values,
    "val_count": val_df[JOINT_LABEL_COLS].sum().astype(int).values,
    "test_count": test_df[JOINT_LABEL_COLS].sum().astype(int).values,
})

split_summary["total_count"] = split_summary[["train_count", "val_count", "test_count"]].sum(axis=1)

SPLIT_SUMMARY_PATH = OUTPUT_DIR / "12_joint_label_split_distribution_colab.csv"

split_summary.to_csv(
    SPLIT_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(split_summary)
print("Saved split distribution:", SPLIT_SUMMARY_PATH)

Train: (2225, 55)
Validation: (278, 55)
Test: (279, 55)


,label,train_count,val_count,test_count,total_count
0,Information_Positive,9,5,1,15
1,Information_Negative,71,5,6,82
2,Product_features_Positive,133,24,18,175
3,Product_features_Negative,109,15,17,141
4,Item_quality_Positive,498,63,67,628
5,Item_quality_Negative,97,11,9,117
6,App_experience_Positive,135,14,21,170
7,App_experience_Negative,44,2,5,51
8,Pricing_Positive,341,26,49,416
9,Pricing_Negative,41,3,5,49


Saved split distribution: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base2\12_joint_label_split_distribution_colab.csv


In [16]:
# ============================================================
# 7. Dataset class
# ============================================================

class TrustpilotJointDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, max_length, text_col, label_cols):
        self.texts = dataframe[text_col].fillna("").astype(str).tolist()
        self.labels = dataframe[label_cols].astype(np.float32).values

        self.encodings = tokenizer(
            self.texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.float32,
        )

        return item

In [17]:
# ============================================================
# 8. Metrics, per-label threshold tuning, and decoding
# ============================================================

def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))


def clean_gpu_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def make_threshold_vector(thresholds, label_cols):
    if isinstance(thresholds, dict):
        return np.array(
            [float(thresholds[label]) for label in label_cols],
            dtype=np.float32,
        )

    threshold_array = np.asarray(thresholds, dtype=np.float32)

    if threshold_array.ndim == 0:
        threshold_array = np.full(
            len(label_cols),
            float(threshold_array),
            dtype=np.float32,
        )

    if len(threshold_array) != len(label_cols):
        raise ValueError(
            f"Threshold length mismatch. Expected {len(label_cols)}, got {len(threshold_array)}."
        )

    return threshold_array


def apply_thresholds_with_conflict_resolution(probs, thresholds, label_cols, aspect_cols):
    threshold_vector = make_threshold_vector(thresholds, label_cols)

    preds = (probs >= threshold_vector.reshape(1, -1)).astype(int)

    label_to_idx = {label: i for i, label in enumerate(label_cols)}
    conflict_count = 0

    for row_idx in range(preds.shape[0]):
        for aspect in aspect_cols:
            pos_label = f"{aspect}_Positive"
            neg_label = f"{aspect}_Negative"

            if pos_label not in label_to_idx or neg_label not in label_to_idx:
                continue

            pos_idx = label_to_idx[pos_label]
            neg_idx = label_to_idx[neg_label]

            if preds[row_idx, pos_idx] == 1 and preds[row_idx, neg_idx] == 1:
                conflict_count += 1

                if probs[row_idx, pos_idx] >= probs[row_idx, neg_idx]:
                    preds[row_idx, neg_idx] = 0
                else:
                    preds[row_idx, pos_idx] = 0

    return preds, conflict_count


def multilabel_metrics(y_true, probs, thresholds, label_cols, aspect_cols):
    y_pred, conflict_count = apply_thresholds_with_conflict_resolution(
        probs=probs,
        thresholds=thresholds,
        label_cols=label_cols,
        aspect_cols=aspect_cols,
    )

    threshold_vector = make_threshold_vector(thresholds, label_cols)

    metrics = {
        "avg_threshold": float(np.mean(threshold_vector)),
        "min_threshold": float(np.min(threshold_vector)),
        "max_threshold": float(np.max(threshold_vector)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "samples_f1": float(f1_score(y_true, y_pred, average="samples", zero_division=0)),
        "micro_precision": float(precision_score(y_true, y_pred, average="micro", zero_division=0)),
        "micro_recall": float(recall_score(y_true, y_pred, average="micro", zero_division=0)),
        "exact_match_accuracy": float(accuracy_score(y_true, y_pred)),
        "hamming_loss": float(hamming_loss(y_true, y_pred)),
        "avg_true_labels_per_review": float(y_true.sum(axis=1).mean()),
        "avg_pred_labels_per_review": float(y_pred.sum(axis=1).mean()),
        "conflict_count_before_resolution": int(conflict_count),
    }

    return metrics, y_pred


def tune_per_label_thresholds(
    y_true,
    probs,
    label_cols,
    aspect_cols,
    thresholds=None,
    default_threshold=0.50,
):
    if thresholds is None:
        thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)

    threshold_rows = []
    best_thresholds = {}

    for label_idx, label in enumerate(label_cols):
        y_label = y_true[:, label_idx]
        p_label = probs[:, label_idx]

        if y_label.sum() == 0:
            best_thresholds[label] = float(default_threshold)

            threshold_rows.append({
                "label": label,
                "best_threshold": float(default_threshold),
                "label_f1": 0.0,
                "label_precision": 0.0,
                "label_recall": 0.0,
                "support": int(y_label.sum()),
                "note": "no_positive_validation_sample",
            })

            continue

        best_score = None
        best_row = None

        for threshold in thresholds:
            y_label_pred = (p_label >= threshold).astype(int)

            label_f1 = float(f1_score(y_label, y_label_pred, zero_division=0))
            label_precision = float(precision_score(y_label, y_label_pred, zero_division=0))
            label_recall = float(recall_score(y_label, y_label_pred, zero_division=0))

            score_tuple = (label_f1, label_precision, label_recall)

            if best_score is None or score_tuple > best_score:
                best_score = score_tuple

                best_row = {
                    "label": label,
                    "best_threshold": float(threshold),
                    "label_f1": label_f1,
                    "label_precision": label_precision,
                    "label_recall": label_recall,
                    "support": int(y_label.sum()),
                    "note": "ok",
                }

        if best_row is None:
            best_thresholds[label] = float(default_threshold)
        else:
            best_thresholds[label] = float(best_row["best_threshold"])
            threshold_rows.append(best_row)

    tuned_metrics, tuned_preds = multilabel_metrics(
        y_true=y_true,
        probs=probs,
        thresholds=best_thresholds,
        label_cols=label_cols,
        aspect_cols=aspect_cols,
    )

    threshold_df = pd.DataFrame(threshold_rows)

    return tuned_metrics, tuned_preds, best_thresholds, threshold_df


def decode_joint_labels(binary_row, label_cols):
    decoded = []

    for value, label in zip(binary_row, label_cols):
        if int(value) == 1:
            aspect, sentiment = label.rsplit("_", 1)
            decoded.append(f"{aspect}:{sentiment}")

    return "; ".join(decoded)


def make_prediction_dataframe(dataframe, y_true, y_pred, probs, label_cols, output_path):
    keep_cols = [
        c for c in ["Card_id", "shop", "Rating", "General_text", "text_clean"]
        if c in dataframe.columns
    ]

    pred_df = dataframe[keep_cols].copy()

    pred_df["true_aspect_sentiment"] = [
        decode_joint_labels(row, label_cols)
        for row in y_true
    ]

    pred_df["pred_aspect_sentiment"] = [
        decode_joint_labels(row, label_cols)
        for row in y_pred
    ]

    for idx, label in enumerate(label_cols):
        pred_df[f"prob_{label}"] = probs[:, idx]

    pred_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )

    return pred_df

In [18]:
# ============================================================
# 9. Custom Trainer and TrainingArguments compatibility helper
# ============================================================

class WeightedMultilabelTrainer(Trainer):
    def __init__(self, *args, pos_weight_tensor=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight_tensor = pos_weight_tensor

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        _ = num_items_in_batch

        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        if self.pos_weight_tensor is not None:
            loss_fct = nn.BCEWithLogitsLoss(
                pos_weight=self.pos_weight_tensor.to(
                    device=logits.device,
                    dtype=torch.float32,
                )
            )
        else:
            loss_fct = nn.BCEWithLogitsLoss()

        loss = loss_fct(logits.float(), labels.float())

        return (loss, outputs) if return_outputs else loss


def make_pos_weight(train_dataframe, label_cols, cap=20.0):
    y = train_dataframe[label_cols].astype(np.float32).values

    pos = y.sum(axis=0)
    neg = y.shape[0] - pos

    weights = []

    for positive_count, negative_count in zip(pos, neg):
        if positive_count <= 0:
            weights.append(1.0)
        else:
            weight = float(negative_count / positive_count)
            weights.append(min(weight, cap))

    return torch.tensor(weights, dtype=torch.float32)


def make_training_args(**kwargs):
    '''
    Makes the notebook compatible with both:
    - Transformers 4.x: evaluation_strategy
    - Transformers 5.x: eval_strategy
    '''
    params = inspect.signature(TrainingArguments.__init__).parameters

    if "eval_strategy" in params:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy", "epoch")
    else:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy", "epoch")

    return TrainingArguments(**kwargs)


def load_sequence_classifier(model_name, num_labels, id2label, label2id):
    base_kwargs = dict(
        num_labels=num_labels,
        problem_type="multi_label_classification",
        id2label=id2label,
        label2id=label2id,
    )

    try:
        return AutoModelForSequenceClassification.from_pretrained(
            model_name,
            use_safetensors=True,
            **base_kwargs,
        )
    except Exception as e:
        print("Safetensors loading failed or unavailable. Falling back to default loading.")
        print("Reason:", repr(e))

        return AutoModelForSequenceClassification.from_pretrained(
            model_name,
            **base_kwargs,
        )

In [ ]:
# ============================================================
# 10. Stage-4 reference configuration
# ============================================================


MODEL_NAME = "microsoft/deberta-v3-small"

SELECTION_MODE = "auto_labeling" 

USE_FP16 = False

SKIP_FINISHED_RUNS = True

RUN_CONFIGS = [
    {
        "run_name": "colab_small_lr3e-5_len224_unweighted_epoch8",
        "learning_rate": 3e-5,
        "num_train_epochs": 8,
        "max_length": 224,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "use_pos_weight": False,
        "pos_weight_cap": 20.0,
    },
    {
        "run_name": "colab_small_lr2e-5_len224_unweighted_epoch8",
        "learning_rate": 2e-5,
        "num_train_epochs": 8,
        "max_length": 224,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "use_pos_weight": False,
        "pos_weight_cap": 20.0,
    },
    {
        "run_name": "colab_small_lr2e-5_len192_mildweighted_cap3_epoch10",
        "learning_rate": 2e-5,
        "num_train_epochs": 10,
        "max_length": 192,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "use_pos_weight": True,
        "pos_weight_cap": 3.0,
    },
        {
        "run_name": "colab_small_lr3e-5_len256_unweighted_epoch8",
        "learning_rate": 3e-5,
        "num_train_epochs": 8,
        "max_length": 256,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "use_pos_weight": False,
        "pos_weight_cap": 20.0,
    },
    {
        "run_name": "colab_small_lr2e-5_len256_unweighted_epoch8",
        "learning_rate": 2e-5,
        "num_train_epochs": 8,
        "max_length": 256,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "use_pos_weight": False,
        "pos_weight_cap": 20.0,
    },
    {
        "run_name": "colab_small_lr1e-5_len224_unweighted_epoch10",
        "learning_rate": 1e-5,
        "num_train_epochs": 10,
        "max_length": 224,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "use_pos_weight": False,
        "pos_weight_cap": 20.0,
    },
    {
        "run_name": "colab_small_lr2e-5_len224_mildweighted_cap2_epoch10",
        "learning_rate": 2e-5,
        "num_train_epochs": 10,
        "max_length": 224,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "use_pos_weight": True,
        "pos_weight_cap": 2.0,
    },
    {
        "run_name": "colab_small_lr2e-5_len224_mildweighted_cap3_epoch10",
        "learning_rate": 2e-5,
        "num_train_epochs": 10,
        "max_length": 224,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "use_pos_weight": True,
        "pos_weight_cap": 3.0,
    },
    {
        "run_name": "colab_small_lr3e-5_len192_mildweighted_cap2_epoch8",
        "learning_rate": 3e-5,
        "num_train_epochs": 8,
        "max_length": 192,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "use_pos_weight": True,
        "pos_weight_cap": 2.0,
    },
    {
        "run_name": "colab_small_lr1_5e-5_len224_mildweighted_cap3_epoch12",
        "learning_rate": 1.5e-5,
        "num_train_epochs": 12,
        "max_length": 224,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "weight_decay": 0.01,
        "use_pos_weight": True,
        "pos_weight_cap": 3.0,
    },
]

display(pd.DataFrame(RUN_CONFIGS))

,run_name,learning_rate,num_train_epochs,max_length,per_device_train_batch_size,per_device_eval_batch_size,gradient_accumulation_steps,weight_decay,use_pos_weight,pos_weight_cap
0,colab_small_lr3e-5_len224_unweighted_epoch8,0.000030,8,224,4,8,2,0.01,False,20.0
1,colab_small_lr2e-5_len224_unweighted_epoch8,0.000020,8,224,4,8,2,0.01,False,20.0
2,colab_small_lr2e-5_len192_mildweighted_cap3_ep...,0.000020,10,192,4,8,2,0.01,True,3.0
3,colab_small_lr3e-5_len256_unweighted_epoch8,0.000030,8,256,4,8,2,0.01,False,20.0
4,colab_small_lr2e-5_len256_unweighted_epoch8,0.000020,8,256,4,8,2,0.01,False,20.0
5,colab_small_lr1e-5_len224_unweighted_epoch10,0.000010,10,224,4,8,2,0.01,False,20.0
6,colab_small_lr2e-5_len224_mildweighted_cap2_ep...,0.000020,10,224,4,8,2,0.01,True,2.0
7,colab_small_lr2e-5_len224_mildweighted_cap3_ep...,0.000020,10,224,4,8,2,0.01,True,3.0
8,colab_small_lr3e-5_len192_mildweighted_cap2_ep...,0.000030,8,192,4,8,2,0.01,True,2.0
9,colab_small_lr1_5e-5_len224_mildweighted_cap3_...,0.000015,12,224,4,8,2,0.01,True,3.0


In [ ]:
# ============================================================
# 11. Run Stage-4 tuning
# ============================================================

def get_selection_score(run_result, selection_mode):
    if selection_mode == "macro_f1":
        return (
            run_result["per_label_val_macro_f1"],
            run_result["per_label_val_micro_f1"],
            -run_result["per_label_val_hamming_loss"],
        )

    if selection_mode == "auto_labeling":
        # Best for auto-labeling:
        # stable micro/weighted/samples F1, high exact match, low hamming loss,
        # and predicted label count close to true label count.
        avg_label_gap = abs(
            run_result["per_label_val_avg_pred_labels_per_review"] -
            run_result["per_label_val_avg_true_labels_per_review"]
        )

        return (
            run_result["per_label_val_micro_f1"],
            run_result["per_label_val_weighted_f1"],
            run_result["per_label_val_samples_f1"],
            run_result["per_label_val_exact_match_accuracy"],
            -run_result["per_label_val_hamming_loss"],
            -avg_label_gap,
        )

    raise ValueError(f"Unknown selection_mode: {selection_mode}")


def print_compact_run_header(run_idx, total_runs, run_name, cfg):
    print("\n" + "=" * 90)
    print(
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        f"Run {run_idx}/{total_runs}: {run_name}"
    )
    print(
        f"lr={cfg['learning_rate']} | "
        f"epochs={cfg['num_train_epochs']} | "
        f"max_len={cfg['max_length']} | "
        f"train_bs={cfg['per_device_train_batch_size']} | "
        f"eval_bs={cfg['per_device_eval_batch_size']} | "
        f"grad_acc={cfg['gradient_accumulation_steps']} | "
        f"weighted={cfg['use_pos_weight']} | "
        f"cap={cfg['pos_weight_cap']}"
    )
    print("=" * 90)


def print_compact_metrics(metrics):
    print(
        f"Result | "
        f"micro={metrics['micro_f1']:.4f} | "
        f"macro={metrics['macro_f1']:.4f} | "
        f"weighted={metrics['weighted_f1']:.4f} | "
        f"samples={metrics['samples_f1']:.4f} | "
        f"exact={metrics['exact_match_accuracy']:.4f} | "
        f"hamming={metrics['hamming_loss']:.4f} | "
        f"avg_true={metrics['avg_true_labels_per_review']:.2f} | "
        f"avg_pred={metrics['avg_pred_labels_per_review']:.2f} | "
        f"conflicts={metrics['conflict_count_before_resolution']}"
    )


def run_colab_stage4_tuning():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    id2label = {i: label for i, label in enumerate(JOINT_LABEL_COLS)}
    label2id = {label: i for i, label in id2label.items()}

    result_filename = "24_colab_deberta_base_stage4_results.csv"
    result_path = OUTPUT_DIR / result_filename

    stage_results = []

    if SKIP_FINISHED_RUNS and result_path.exists():
        old_results = pd.read_csv(result_path)
        stage_results = old_results.to_dict("records")
        print(f"Loaded existing results: {result_path}")

    completed_runs = {
        str(row["run_name"])
        for row in stage_results
        if "run_name" in row
    }

    stage_best = None

    # Rebuild current best from existing rows if available.
    for row in stage_results:
        try:
            score = get_selection_score(row, SELECTION_MODE)

            if stage_best is None or score > stage_best["score_tuple"]:
                stage_best = {
                    "score_tuple": score,
                    "run_result": row,
                    "best_model_path": row.get("best_model_path"),
                    "threshold_path": row.get("threshold_path"),
                }
        except Exception:
            pass

    for run_idx, cfg in enumerate(RUN_CONFIGS, start=1):
        clean_gpu_memory()

        run_name = str(cfg["run_name"])
        run_dir = RUNS_DIR / "colab_stage4_base" / run_name
        best_model_path = run_dir / "best_model"
        threshold_path = run_dir / "best_per_label_thresholds.json"

        if (
            SKIP_FINISHED_RUNS and
            run_name in completed_runs and
            best_model_path.exists() and
            threshold_path.exists()
        ):
            print(f"Skipping completed run: {run_name}")
            continue

        print_compact_run_header(
            run_idx=run_idx,
            total_runs=len(RUN_CONFIGS),
            run_name=run_name,
            cfg=cfg,
        )

        run_dir.mkdir(parents=True, exist_ok=True)

        train_dataset = TrustpilotJointDataset(
            train_df,
            tokenizer,
            int(cfg["max_length"]),
            TEXT_COL,
            JOINT_LABEL_COLS,
        )

        val_dataset = TrustpilotJointDataset(
            val_df,
            tokenizer,
            int(cfg["max_length"]),
            TEXT_COL,
            JOINT_LABEL_COLS,
        )

        model = load_sequence_classifier(
            model_name=MODEL_NAME,
            num_labels=len(JOINT_LABEL_COLS),
            id2label=id2label,
            label2id=label2id,
        )

        model.float()

        pos_weight_tensor = None

        if bool(cfg.get("use_pos_weight", False)):
            pos_weight_tensor = make_pos_weight(
                train_df,
                JOINT_LABEL_COLS,
                cap=float(cfg.get("pos_weight_cap", 20.0)),
            )

            pos_weight_df = pd.DataFrame({
                "label": JOINT_LABEL_COLS,
                "pos_weight": pos_weight_tensor.cpu().numpy(),
            })

            pos_weight_df.to_csv(
                run_dir / "pos_weights.csv",
                index=False,
                encoding="utf-8-sig",
            )

        def compute_metrics_fixed_050(eval_pred):
            logits, labels = eval_pred

            if isinstance(logits, tuple):
                logits = logits[0]

            probs = sigmoid_np(logits)

            metrics, _ = multilabel_metrics(
                labels,
                probs,
                thresholds=0.50,
                label_cols=JOINT_LABEL_COLS,
                aspect_cols=ASPECT_COLS,
            )

            return {
                "micro_f1": metrics["micro_f1"],
                "macro_f1": metrics["macro_f1"],
                "weighted_f1": metrics["weighted_f1"],
                "samples_f1": metrics["samples_f1"],
                "hamming_loss": metrics["hamming_loss"],
            }

        training_args = make_training_args(
            output_dir=str(run_dir),
            run_name=run_name,
            learning_rate=float(cfg["learning_rate"]),
            num_train_epochs=int(cfg["num_train_epochs"]),
            per_device_train_batch_size=int(cfg["per_device_train_batch_size"]),
            per_device_eval_batch_size=int(cfg["per_device_eval_batch_size"]),
            gradient_accumulation_steps=int(cfg["gradient_accumulation_steps"]),
            weight_decay=float(cfg["weight_decay"]),

            evaluation_strategy="epoch",
            save_strategy="epoch",

            # Cleaner Colab output
            logging_strategy="epoch",
            disable_tqdm=False,

            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="eval_macro_f1",
            greater_is_better=True,

            fp16=bool(USE_FP16),
            bf16=False,

            gradient_checkpointing=True,
            eval_accumulation_steps=8,

            report_to="none",
            seed=SEED,
            data_seed=SEED,
            remove_unused_columns=True,
        )

        trainer = WeightedMultilabelTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics_fixed_050,
            pos_weight_tensor=pos_weight_tensor,
        )

        trainer.train()

        val_pred_output = trainer.predict(val_dataset)
        val_logits = val_pred_output.predictions

        if isinstance(val_logits, tuple):
            val_logits = val_logits[0]

        val_probs = sigmoid_np(val_logits)
        val_true = val_df[JOINT_LABEL_COLS].astype(int).values

        best_per_label_metrics, val_pred_per_label, per_label_thresholds, per_label_threshold_table = tune_per_label_thresholds(
            y_true=val_true,
            probs=val_probs,
            label_cols=JOINT_LABEL_COLS,
            aspect_cols=ASPECT_COLS,
        )

        per_label_threshold_table.to_csv(
            run_dir / "threshold_tuning_validation_per_label.csv",
            index=False,
            encoding="utf-8-sig",
        )

        with open(threshold_path, "w", encoding="utf-8") as f:
            json.dump(per_label_thresholds, f, indent=2)

        trainer.save_model(str(best_model_path))
        tokenizer.save_pretrained(str(best_model_path))

        run_result = {
            **cfg,
            "stage": "colab_stage4_base",
            "model_name": MODEL_NAME,
            "selection_mode": SELECTION_MODE,
            "run_name": run_name,

            "per_label_avg_threshold": best_per_label_metrics["avg_threshold"],
            "per_label_min_threshold": best_per_label_metrics["min_threshold"],
            "per_label_max_threshold": best_per_label_metrics["max_threshold"],
            "per_label_val_micro_f1": best_per_label_metrics["micro_f1"],
            "per_label_val_macro_f1": best_per_label_metrics["macro_f1"],
            "per_label_val_weighted_f1": best_per_label_metrics["weighted_f1"],
            "per_label_val_samples_f1": best_per_label_metrics["samples_f1"],
            "per_label_val_exact_match_accuracy": best_per_label_metrics["exact_match_accuracy"],
            "per_label_val_hamming_loss": best_per_label_metrics["hamming_loss"],
            "per_label_val_avg_true_labels_per_review": best_per_label_metrics["avg_true_labels_per_review"],
            "per_label_val_avg_pred_labels_per_review": best_per_label_metrics["avg_pred_labels_per_review"],
            "per_label_val_conflict_count_before_resolution": best_per_label_metrics["conflict_count_before_resolution"],

            "run_dir": str(run_dir),
            "best_model_path": str(best_model_path),
            "threshold_path": str(threshold_path),
        }

        stage_results.append(run_result)

        print_compact_metrics(best_per_label_metrics)

        score_tuple = get_selection_score(run_result, SELECTION_MODE)

        if stage_best is None or score_tuple > stage_best["score_tuple"]:
            stage_best = {
                "score_tuple": score_tuple,
                "run_result": run_result,
                "best_model_path": str(best_model_path),
                "threshold_path": str(threshold_path),
            }

            if BEST_MODEL_DIR.exists():
                shutil.rmtree(BEST_MODEL_DIR)

            shutil.copytree(best_model_path, BEST_MODEL_DIR)

            shutil.copy2(
                threshold_path,
                OUTPUT_DIR / "colab_best_per_label_thresholds.json",
            )

            print("✅ New best model copied to:", BEST_MODEL_DIR)

        stage_results_df = pd.DataFrame(stage_results)

        stage_results_df.to_csv(
            result_path,
            index=False,
            encoding="utf-8-sig",
        )

        del trainer
        del model
        del train_dataset
        del val_dataset
        del val_pred_output

        clean_gpu_memory()

    assert stage_best is not None, "No Colab Stage-4 run was completed."

    final_results_df = pd.DataFrame(stage_results)

    print("\nColab Stage-4 tuning finished.")
    print("Selection mode:", SELECTION_MODE)
    print("Best run:", stage_best["run_result"]["run_name"])
    print("Best model path:", stage_best["best_model_path"])
    print("Best threshold path:", stage_best["threshold_path"])
    print("Copied best model dir:", BEST_MODEL_DIR)

    print("\nFinal Colab Stage-4 ranking:")
    display(
        final_results_df.sort_values(
            [
                "per_label_val_micro_f1",
                "per_label_val_weighted_f1",
                "per_label_val_hamming_loss",
            ],
            ascending=[False, False, True],
        )
    )

    return stage_best, final_results_df


colab_stage4_best, colab_stage4_results_df = run_colab_stage4_tuning()


[2026-06-24 14:35:40] Run 1/10: colab_small_lr3e-5_len224_unweighted_epoch8
lr=3e-05 | epochs=8 | max_len=224 | train_bs=4 | eval_bs=8 | grad_acc=2 | weighted=False | cap=20.0


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.de

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Weighted F1,Samples F1,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.529517,0.191108,0.469008,0.150837,0.386651,0.346554,0.071112,2.718500,102.260000,12.875000
2,0.347793,0.147292,0.612033,0.215025,0.525587,0.580434,0.051743,2.720900,102.174000,12.864000
3,0.279885,0.124229,0.677259,0.288750,0.594553,0.666217,0.044964,2.719500,102.224000,12.870000
4,0.228485,0.117918,0.700000,0.301186,0.619962,0.692926,0.042335,2.727100,101.939000,12.834000
5,0.193182,0.104826,0.771710,0.435491,0.733157,0.764415,0.035279,2.731200,101.785000,12.815000
6,0.167374,0.100855,0.764706,0.412836,0.717858,0.761072,0.035418,2.728300,101.893000,12.828000
7,0.149360,0.097326,0.777070,0.424544,0.733884,0.762573,0.033896,2.728500,101.887000,12.828000
8,0.139551,0.096492,0.781818,0.441537,0.744516,0.765430,0.033204,2.743400,101.332000,12.758000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Result | micro=0.7330 | macro=0.5536 | weighted=0.7858 | samples=0.7723 | exact=0.4568 | hamming=0.0494 | avg_true=2.12 | avg_pred=2.69 | conflicts=8

Colab Stage-4 tuning finished.
Selection mode: auto_labeling
Best run: colab_small_lr3e-5_len256_unweighted_epoch8
Best model path: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base2\tuning_runs2\colab_stage4_base\colab_small_lr3e-5_len256_unweighted_epoch8\best_model
Best threshold path: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base2\tuning_runs2\colab_stage4_base\colab_small_lr3e-5_len256_unweighted_epoch8\best_per_label_thresholds.json
Copied best model dir: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base2\colab_best_joint_deberta_model2

Final Colab Stage-4 ranking:


,run_name,learning_rate,num_train_epochs,max_length,per_device_train_batch_size,per_device_eval_batch_size,gradient_accumulation_steps,weight_decay,use_pos_weight,pos_weight_cap,...,per_label_val_weighted_f1,per_label_val_samples_f1,per_label_val_exact_match_accuracy,per_label_val_hamming_loss,per_label_val_avg_true_labels_per_review,per_label_val_avg_pred_labels_per_review,per_label_val_conflict_count_before_resolution,run_dir,best_model_path,threshold_path
3,colab_small_lr3e-5_len256_unweighted_epoch8,0.000030,8,256,4,8,2,0.01,False,20.0,...,0.809656,0.821444,0.589928,0.032512,2.122302,2.284173,1,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...
0,colab_small_lr3e-5_len224_unweighted_epoch8,0.000030,8,224,4,8,2,0.01,False,20.0,...,0.809349,0.825412,0.579137,0.032928,2.122302,2.309353,1,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...
8,colab_small_lr3e-5_len192_mildweighted_cap2_ep...,0.000030,8,192,4,8,2,0.01,True,2.0,...,0.804246,0.794685,0.507194,0.042197,2.122302,2.564748,3,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...
7,colab_small_lr2e-5_len224_mildweighted_cap3_ep...,0.000020,10,224,4,8,2,0.01,True,3.0,...,0.801421,0.799811,0.539568,0.042889,2.122302,2.553957,9,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...
6,colab_small_lr2e-5_len224_mildweighted_cap2_ep...,0.000020,10,224,4,8,2,0.01,True,2.0,...,0.793822,0.796970,0.528777,0.043027,2.122302,2.543165,5,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...
1,colab_small_lr2e-5_len224_unweighted_epoch8,0.000020,8,224,4,8,2,0.01,False,20.0,...,0.758017,0.766911,0.456835,0.042750,2.122302,2.348921,3,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...
4,colab_small_lr2e-5_len256_unweighted_epoch8,0.000020,8,256,4,8,2,0.01,False,20.0,...,0.762018,0.777079,0.478417,0.043165,2.122302,2.366906,1,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...
2,colab_small_lr2e-5_len192_mildweighted_cap3_ep...,0.000020,10,192,4,8,2,0.01,True,3.0,...,0.798585,0.790824,0.482014,0.046348,2.122302,2.658273,5,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...
5,colab_small_lr1e-5_len224_unweighted_epoch10,0.000010,10,224,4,8,2,0.01,False,20.0,...,0.726817,0.745066,0.438849,0.044272,2.122302,2.223022,0,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...
9,colab_small_lr1_5e-5_len224_mildweighted_cap3_...,0.000015,12,224,4,8,2,0.01,True,3.0,...,0.785823,0.772327,0.456835,0.049391,2.122302,2.687050,8,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...,C:\Users\Lenovo\Documents\Skripsi\dataset\phas...


In [ ]:
# ============================================================
# 11B. Force/check selected best model for test evaluation and inference
# ============================================================

SELECTED_RUN_NAME = "colab_small_lr3e-5_len256_unweighted_epoch8"

selected_run_dir = RUNS_DIR / "colab_stage4_base" / SELECTED_RUN_NAME
selected_model_dir = selected_run_dir / "best_model"
selected_threshold_path = selected_run_dir / "best_per_label_thresholds.json"

if not selected_model_dir.exists():
    raise FileNotFoundError(f"Selected model folder not found: {selected_model_dir}")

if not selected_threshold_path.exists():
    raise FileNotFoundError(f"Selected threshold file not found: {selected_threshold_path}")

if BEST_MODEL_DIR.exists():
    shutil.rmtree(BEST_MODEL_DIR)

shutil.copytree(selected_model_dir, BEST_MODEL_DIR)

shutil.copy2(
    selected_threshold_path,
    OUTPUT_DIR / "colab_best_per_label_thresholds.json",
)

selected_info = {
    "selected_run_name": SELECTED_RUN_NAME,
    "selected_model_dir": str(selected_model_dir),
    "selected_threshold_path": str(selected_threshold_path),
    "copied_best_model_dir": str(BEST_MODEL_DIR),
    "copied_threshold_path": str(OUTPUT_DIR / "colab_best_per_label_thresholds.json"),
    "model_name": MODEL_NAME,
}

with open(OUTPUT_DIR / "selected_best_model_info.json", "w", encoding="utf-8") as f:
    json.dump(selected_info, f, indent=2)

print("✅ Forced selected best model for Section 12 and Section 13")
print(json.dumps(selected_info, indent=2))

✅ Forced selected best model for Section 12 and Section 13
{
  "selected_run_name": "colab_small_lr3e-5_len256_unweighted_epoch8",
  "selected_model_dir": "C:\\Users\\Lenovo\\Documents\\Skripsi\\dataset\\phase1\\trustpilot_reformatted_output\\deberta_joint_aspect_sentiment_output_colab_base2\\tuning_runs2\\colab_stage4_base\\colab_small_lr3e-5_len256_unweighted_epoch8\\best_model",
  "selected_threshold_path": "C:\\Users\\Lenovo\\Documents\\Skripsi\\dataset\\phase1\\trustpilot_reformatted_output\\deberta_joint_aspect_sentiment_output_colab_base2\\tuning_runs2\\colab_stage4_base\\colab_small_lr3e-5_len256_unweighted_epoch8\\best_per_label_thresholds.json",
  "copied_best_model_dir": "C:\\Users\\Lenovo\\Documents\\Skripsi\\dataset\\phase1\\trustpilot_reformatted_output\\deberta_joint_aspect_sentiment_output_colab_base2\\colab_best_joint_deberta_model2",
  "copied_threshold_path": "C:\\Users\\Lenovo\\Documents\\Skripsi\\dataset\\phase1\\trustpilot_reformatted_output\\deberta_joint_aspect_

In [22]:
# ============================================================
# 12. Evaluate the selected best model on the test set
# ============================================================

BEST_THRESHOLD_PATH = OUTPUT_DIR / "colab_best_per_label_thresholds.json"

if not BEST_MODEL_DIR.exists():
    raise FileNotFoundError(f"Best model folder not found: {BEST_MODEL_DIR}")

if not BEST_THRESHOLD_PATH.exists():
    raise FileNotFoundError(f"Best threshold file not found: {BEST_THRESHOLD_PATH}")

with open(BEST_THRESHOLD_PATH, "r", encoding="utf-8") as f:
    best_thresholds = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(str(BEST_MODEL_DIR))

model = AutoModelForSequenceClassification.from_pretrained(
    str(BEST_MODEL_DIR)
)

model.to(DEVICE)
model.float()

test_dataset = TrustpilotJointDataset(
    test_df,
    tokenizer,
    max_length=max(int(cfg["max_length"]) for cfg in RUN_CONFIGS),
    text_col=TEXT_COL,
    label_cols=JOINT_LABEL_COLS,
)

eval_args = make_training_args(
    output_dir=str(OUTPUT_DIR / "test_eval_tmp"),
    per_device_eval_batch_size=4,
    evaluation_strategy="no",
    save_strategy="no",
    report_to="none",
    fp16=False,
    bf16=False,
    eval_accumulation_steps=8,
    remove_unused_columns=True,
)

test_trainer = WeightedMultilabelTrainer(
    model=model,
    args=eval_args,
)

test_pred_output = test_trainer.predict(test_dataset)
test_logits = test_pred_output.predictions

if isinstance(test_logits, tuple):
    test_logits = test_logits[0]

test_probs = sigmoid_np(test_logits)
test_true = test_df[JOINT_LABEL_COLS].astype(int).values

test_metrics, test_pred = multilabel_metrics(
    y_true=test_true,
    probs=test_probs,
    thresholds=best_thresholds,
    label_cols=JOINT_LABEL_COLS,
    aspect_cols=ASPECT_COLS,
)

TEST_METRICS_PATH = OUTPUT_DIR / "25_colab_best_test_metrics.json"

with open(TEST_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

TEST_PREDICTIONS_PATH = OUTPUT_DIR / "26_colab_best_test_predictions.csv"

test_pred_df = make_prediction_dataframe(
    dataframe=test_df,
    y_true=test_true,
    y_pred=test_pred,
    probs=test_probs,
    label_cols=JOINT_LABEL_COLS,
    output_path=TEST_PREDICTIONS_PATH,
)

report_dict = classification_report(
    test_true,
    test_pred,
    target_names=JOINT_LABEL_COLS,
    zero_division=0,
    output_dict=True,
)

report_df = pd.DataFrame(report_dict).transpose()

REPORT_PATH = OUTPUT_DIR / "27_colab_best_test_classification_report_per_label.csv"

report_df.to_csv(
    REPORT_PATH,
    index=True,
    encoding="utf-8-sig",
)

print("Test metrics:")
print(json.dumps(test_metrics, indent=2))

print("Saved test metrics:", TEST_METRICS_PATH)
print("Saved test predictions:", TEST_PREDICTIONS_PATH)
print("Saved per-label report:", REPORT_PATH)

display(report_df)
display(test_pred_df.head(10))

del test_trainer
del model
del test_dataset
clean_gpu_memory()

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Test metrics:
{
  "avg_threshold": 0.32884615659713745,
  "min_threshold": 0.10000000149011612,
  "max_threshold": 0.699999988079071,
  "micro_f1": 0.7681498829039812,
  "macro_f1": 0.5459527889060121,
  "weighted_f1": 0.7762465589918314,
  "samples_f1": 0.7691795963775724,
  "micro_precision": 0.7651632970451011,
  "micro_recall": 0.7711598746081505,
  "exact_match_accuracy": 0.45878136200716846,
  "hamming_loss": 0.04094292803970223,
  "avg_true_labels_per_review": 2.2867383512544803,
  "avg_pred_labels_per_review": 2.304659498207885,
  "conflict_count_before_resolution": 0
}
Saved test metrics: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base2\25_colab_best_test_metrics.json
Saved test predictions: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base2\26_colab_best_test_predictions.csv
Saved per-label report: C:\Users\Lenovo\Docume

,precision,recall,f1-score,support
Information_Positive,0.000000,0.000000,0.000000,1.0
Information_Negative,0.333333,0.666667,0.444444,6.0
Product_features_Positive,0.681818,0.833333,0.750000,18.0
Product_features_Negative,0.615385,0.470588,0.533333,17.0
Item_quality_Positive,0.816901,0.865672,0.840580,67.0
Item_quality_Negative,0.388889,0.777778,0.518519,9.0
App_experience_Positive,0.800000,0.571429,0.666667,21.0
App_experience_Negative,0.000000,0.000000,0.000000,5.0
Pricing_Positive,0.975610,0.816327,0.888889,49.0
Pricing_Negative,0.272727,0.600000,0.375000,5.0


,Card_id,shop,Rating,General_text,text_clean,true_aspect_sentiment,pred_aspect_sentiment,prob_Information_Positive,prob_Information_Negative,prob_Product_features_Positive,...,prob_Shipping_Positive,prob_Shipping_Negative,prob_Return_process_Positive,prob_Return_process_Negative,prob_Refund_process_Positive,prob_Refund_process_Negative,prob_Customer_service_Positive,prob_Customer_service_Negative,prob_Trust_Positive,prob_Trust_Negative
0,601aa2aa679d9705c862c31d,Boozt,5,Positive,I am quite satisfied!. I am quite satisfied wi...,Delivered_product_status:Positive,Item_quality:Positive,0.000296,0.000476,0.072592,...,0.183120,0.007673,0.000425,0.000643,0.000309,0.001933,0.003085,0.002448,0.003767,0.000803
1,60bf8a3af9f4870a0ca67e8d,Boozt,1,Negative,I received a wrong size item. I received a wro...,Delivered_product_status:Negative; Customer_se...,Customer_service:Negative,0.001485,0.038561,0.006597,...,0.002351,0.292313,0.001615,0.085616,0.006637,0.114583,0.019411,0.966806,0.004149,0.080540
2,5e6f96a93c93ae096465070e,Sheinside,5,Negative,*** Updated*** WASN'T SENT THE RIGHT ITEM,Delivered_product_status:Negative,Item_quality:Negative; Delivered_product_statu...,0.005729,0.134028,0.014695,...,0.006742,0.942618,0.003394,0.068691,0.024130,0.048121,0.001949,0.036321,0.004571,0.172462
3,5a182a9ac0c9900ddc592715,Zalando,5,Mixed,very quick service from zalando.. very quick s...,Shipping:Negative,Shipping:Positive,0.000173,0.000783,0.004374,...,0.955458,0.014951,0.003757,0.000585,0.001035,0.001213,0.005862,0.000630,0.010497,0.000648
4,5996f391a64b4105d835567e,Zalando,5,Positive,Superfast delivery and all very easy. Superfas...,Delivered_product_status:Negative; Shipping:Po...,App_experience:Positive; Shipping:Positive; Re...,0.044008,0.022785,0.177825,...,0.997383,0.005732,0.876070,0.027729,0.144310,0.022403,0.042389,0.003908,0.083962,0.006997
5,5f61ed6e02e8570a48735c77,Sheinside,3,Mixed,Problems tracking. Ive seen some other reviews...,Item_quality:Positive; Shipping:Negative,Item_quality:Positive; Shipping:Negative,0.000200,0.002460,0.003318,...,0.019224,0.896515,0.000287,0.000801,0.001280,0.002165,0.004294,0.006175,0.020728,0.002286
6,5931662820b79f0a909ad607,Zalando,5,Positive,I received 10% off my first order-which …. I r...,Item_quality:Positive; Pricing:Positive; Shipp...,Item_quality:Positive; Shipping:Positive; Trus...,0.001833,0.000710,0.133901,...,0.993667,0.000789,0.011262,0.000487,0.000938,0.000692,0.011244,0.000149,0.961203,0.000216
7,605e3a5ff85d750870376e61,Wish,1,Negative,"Very poor quality product, service and tracking",Item_quality:Negative; Shipping:Negative,Product_features:Negative; Item_quality:Negati...,0.003591,0.039830,0.034808,...,0.006165,0.038264,0.002944,0.033083,0.010367,0.024032,0.005385,0.033530,0.006013,0.070864
8,5a1838edc0c9900ddc592c34,Zalando,4,Mixed,I would have given 5* if my delivery …. I woul...,Shipping:Negative; Refund_process:Positive; Cu...,Shipping:Negative,0.003990,0.041935,0.005283,...,0.185265,0.828955,0.110302,0.058620,0.592487,0.059391,0.224565,0.011677,0.012458,0.003087
9,5476c1bd0000640002b75994,Nelly,5,Positive,Great service and good products!. Modern and n...,Item_quality:Positive; App_experience:Positive...,Item_quality:Positive; Trust:Positive,0.001972,0.001887,0.111448,...,0.035882,0.001702,0.004784,0.002018,0.002028,0.001582,0.056261,0.007103,0.715829,0.001029


In [ ]:
# ============================================================
# 13. Inference helper for the next auto-labeling step
# ============================================================


def predict_joint_absa_texts(
    texts,
    model_dir=BEST_MODEL_DIR,
    threshold_path=OUTPUT_DIR / "colab_best_per_label_thresholds.json",
    max_length=224,
    batch_size=8,
):
    with open(threshold_path, "r", encoding="utf-8") as f:
        thresholds = json.load(f)

    tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
    model.to(DEVICE)
    model.eval()

    all_probs = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )

        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.sigmoid(outputs.logits).detach().cpu().numpy()

        all_probs.append(probs)

    probs = np.vstack(all_probs)

    preds, conflict_count = apply_thresholds_with_conflict_resolution(
        probs=probs,
        thresholds=thresholds,
        label_cols=JOINT_LABEL_COLS,
        aspect_cols=ASPECT_COLS,
    )

    compact_predictions = [
        decode_joint_labels(row, JOINT_LABEL_COLS)
        for row in preds
    ]

    result_df = pd.DataFrame({
        "text": texts,
        "pred_aspect_sentiment": compact_predictions,
        "conflict_count_info": conflict_count,
    })

    for idx, label in enumerate(JOINT_LABEL_COLS):
        result_df[f"prob_{label}"] = probs[:, idx]

    return result_df


# Example:
sample_texts = [
    "The shipping was fast and the product quality was excellent.",
    "Customer service ignored my refund request.",
]

sample_pred_df = predict_joint_absa_texts(sample_texts)
display(sample_pred_df)

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

,text,pred_aspect_sentiment,conflict_count_info,prob_Information_Positive,prob_Information_Negative,prob_Product_features_Positive,prob_Product_features_Negative,prob_Item_quality_Positive,prob_Item_quality_Negative,prob_App_experience_Positive,...,prob_Shipping_Positive,prob_Shipping_Negative,prob_Return_process_Positive,prob_Return_process_Negative,prob_Refund_process_Positive,prob_Refund_process_Negative,prob_Customer_service_Positive,prob_Customer_service_Negative,prob_Trust_Positive,prob_Trust_Negative
0,The shipping was fast and the product quality ...,Item_quality:Positive; Shipping:Positive,0,0.000311,0.000375,0.043800,0.002092,0.962655,0.000784,0.004408,...,0.943833,0.00222,0.001217,0.000696,0.000886,0.000701,0.005693,0.001455,0.011837,0.000940
1,Customer service ignored my refund request.,Customer_service:Negative,0,0.001133,0.013664,0.002183,0.006045,0.009096,0.014928,0.004519,...,0.013025,0.02454,0.002479,0.046372,0.004083,0.097842,0.022001,0.951999,0.004565,0.070496
